# Milestone 3 - CNN vs RNN

# Milestone 3: Deep Architectures and Training

## What Makes a Network Deep

Depth means that a network has multiple hidden transformation layers between input and output. Deeper models can learn hierarchical representations. In an image task, early layers may learn small edges or strokes, while later layers combine them into digit-like shapes. Depth increases model capacity, but it also makes training harder because gradients must travel through more operations.

## Training Challenges

The main issues I focused on were vanishing and exploding gradients. Vanishing gradients make early layers learn slowly. Exploding gradients make updates unstable. I used ReLU in the CNN, gated recurrence in the GRU model, Adam optimization, gradient clipping, dropout, batch normalization, learning-rate scheduling, and early stopping.

## Two Architectures

I used the scikit-learn handwritten digits dataset because it is small enough to train quickly but still real enough to compare architectures.

Model 1: CNN

- Treats each digit as an 8x8 image.
- Uses two convolutional blocks with batch normalization and ReLU.
- Uses dropout before the final classifier.
- Best fit for the task because the spatial structure of the image matters.

Model 2: RNN

- Treats each digit image as a sequence of eight rows.
- Uses a GRU to process row-by-row information.
- Uses dropout before classification.
- Less natural for images, but useful as a comparison because it shows how architecture assumptions affect performance.

## Training Process

The script creates a train, validation, and test split. It saves:

- `artifacts/milestone3_metrics.json`
- `artifacts/milestone3_digits_training.png`

I expected the CNN to perform better because nearby pixels in a digit are meaningful. The RNN has to infer spatial structure indirectly from row order, so it is solving the same task with a weaker assumption.

## Evaluation

The CNN is the better design for this dataset because it matches the data geometry. The RNN is still trainable, but it is not using the two-dimensional structure as directly. This comparison helped clarify that model choice is not just about raw complexity. It is about matching the architecture to the structure of the data.


## Runnable Code:

In [6]:
"""
Milestone 3: Compare two deeper architectures on digit images.

A small
CNN uses spatial locality directly. A small RNN treats each image as a sequence
of eight rows, which is less natural but useful for comparing model assumptions.
"""

from __future__ import annotations

import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIG_DIR = PROJECT_ROOT / ".matplotlib"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed: int = 12) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)


def make_loaders(batch_size: int = 64) -> tuple[DataLoader, DataLoader, DataLoader]:
    digits = load_digits()
    x = digits.images.astype("float32") / 16.0
    y = digits.target.astype("int64")

    x_train, x_temp, y_train, y_temp = train_test_split(
        x, y, test_size=0.30, stratify=y, random_state=12
    )
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=12
    )

    def loader(features: np.ndarray, labels: np.ndarray, shuffle: bool) -> DataLoader:
        dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))
        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return loader(x_train, y_train, True), loader(x_val, y_val, False), loader(x_test, y_test, False)


class DigitCNN(nn.Module):
    """Convolutional model with batch normalization and dropout."""

    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(32 * 4 * 4, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.unsqueeze(1))


class RowRNN(nn.Module):
    """Recurrent model that sees each image row as a timestep."""

    def __init__(self) -> None:
        super().__init__()
        self.rnn = nn.GRU(input_size=8, hidden_size=32, batch_first=True)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(32, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, hidden = self.rnn(x)
        return self.classifier(self.dropout(hidden[-1]))


def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float, np.ndarray, np.ndarray]:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    losses: list[float] = []
    predictions: list[int] = []
    targets: list[int] = []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            losses.append(float(loss.item()))
            predictions.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            targets.extend(y_batch.cpu().numpy().tolist())
    return float(np.mean(losses)), accuracy_score(targets, predictions), np.array(targets), np.array(predictions)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int = 8,
    patience: int = 3,
    model_name: str = "model",
) -> tuple[nn.Module, dict[str, list[float]]]:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    best_state = None
    best_val_loss = float("inf")
    stale_epochs = 0

    for _ in range(max_epochs):
        model.train()
        batch_losses: list[float] = []
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            batch_losses.append(float(loss.item()))

        scheduler.step()
        val_loss, val_accuracy, _, _ = evaluate(model, val_loader)
        history["train_loss"].append(float(np.mean(batch_losses)))
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(float(val_accuracy))
        print(
            f"{model_name} epoch {len(history['train_loss'])}: "
            f"train_loss={history['train_loss'][-1]:.4f}, "
            f"val_loss={val_loss:.4f}, val_acc={val_accuracy:.3f}",
            flush=True,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def plot_histories(histories: dict[str, dict[str, list[float]]]) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for name, history in histories.items():
        axes[0].plot(history["train_loss"], label=f"{name} train")
        axes[0].plot(history["val_loss"], label=f"{name} val")
        axes[1].plot(history["val_accuracy"], label=name)
    axes[0].set_title("Loss during training")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy")
    axes[0].legend()
    axes[1].set_title("Validation accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0.7, 1.0)
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "milestone3_digits_training.png", dpi=160)
    plt.close(fig)


def main() -> dict[str, object]:
    set_seed()
    train_loader, val_loader, test_loader = make_loaders()
    models: dict[str, nn.Module] = {"CNN": DigitCNN(), "RNN": RowRNN()}
    histories: dict[str, dict[str, list[float]]] = {}
    metrics: dict[str, object] = {}

    for name, model in models.items():
        trained, history = train_model(model, train_loader, val_loader, model_name=name)
        test_loss, test_accuracy, y_true, y_pred = evaluate(trained, test_loader)
        histories[name] = history
        metrics[name] = {
            "epochs_ran": len(history["train_loss"]),
            "test_loss": round(test_loss, 4),
            "test_accuracy": round(float(test_accuracy), 4),
            "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        }

    plot_histories(histories)
    with open(ARTIFACT_DIR / "milestone3_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))
    return metrics


main()


CNN epoch 1: train_loss=0.6700, val_loss=0.3791, val_acc=0.922
CNN epoch 2: train_loss=0.1306, val_loss=0.1817, val_acc=0.926
CNN epoch 3: train_loss=0.0830, val_loss=0.0731, val_acc=0.959
CNN epoch 4: train_loss=0.0384, val_loss=0.0877, val_acc=0.959
CNN epoch 5: train_loss=0.0209, val_loss=0.0800, val_acc=0.967
CNN epoch 6: train_loss=0.0140, val_loss=0.0559, val_acc=0.981
CNN epoch 7: train_loss=0.0209, val_loss=0.0689, val_acc=0.970
CNN epoch 8: train_loss=0.0101, val_loss=0.0792, val_acc=0.970
RNN epoch 1: train_loss=2.1302, val_loss=1.6432, val_acc=0.496
RNN epoch 2: train_loss=1.3513, val_loss=0.9181, val_acc=0.685
RNN epoch 3: train_loss=0.8402, val_loss=0.5915, val_acc=0.796
RNN epoch 4: train_loss=0.5793, val_loss=0.3917, val_acc=0.870
RNN epoch 5: train_loss=0.4031, val_loss=0.2947, val_acc=0.911
RNN epoch 6: train_loss=0.3267, val_loss=0.2529, val_acc=0.922
RNN epoch 7: train_loss=0.2539, val_loss=0.2062, val_acc=0.933
RNN epoch 8: train_loss=0.2311, val_loss=0.2190, val_ac

{'CNN': {'epochs_ran': 8,
  'test_loss': 0.0523,
  'test_accuracy': 0.9889,
  'confusion_matrix': [[27, 0, 0, 0, 0, 0, 0, 0, 0, 0],
   [0, 26, 0, 0, 0, 0, 0, 0, 1, 0],
   [0, 0, 26, 0, 0, 0, 0, 0, 0, 0],
   [0, 0, 0, 28, 0, 0, 0, 0, 0, 0],
   [0, 0, 0, 0, 27, 0, 0, 0, 0, 0],
   [1, 0, 0, 0, 0, 26, 1, 0, 0, 0],
   [0, 0, 0, 0, 0, 0, 27, 0, 0, 0],
   [0, 0, 0, 0, 0, 0, 0, 27, 0, 0],
   [0, 0, 0, 0, 0, 0, 0, 0, 26, 0],
   [0, 0, 0, 0, 0, 0, 0, 0, 0, 27]]},
 'RNN': {'epochs_ran': 8,
  'test_loss': 0.1542,
  'test_accuracy': 0.9593,
  'confusion_matrix': [[27, 0, 0, 0, 0, 0, 0, 0, 0, 0],
   [0, 24, 0, 2, 0, 1, 0, 0, 0, 0],
   [0, 0, 25, 0, 0, 0, 0, 0, 1, 0],
   [0, 0, 0, 27, 0, 0, 0, 0, 0, 1],
   [0, 0, 0, 0, 27, 0, 0, 0, 0, 0],
   [0, 0, 0, 0, 0, 27, 1, 0, 0, 0],
   [0, 0, 0, 0, 0, 0, 27, 0, 0, 0],
   [0, 0, 0, 0, 0, 0, 0, 27, 0, 0],
   [0, 2, 2, 0, 0, 1, 0, 0, 21, 0],
   [0, 0, 0, 0, 0, 0, 0, 0, 0, 27]]}}